# Orders Profiling

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/olist_orders_dataset.csv')
print('Rows, Cols:', df.shape)
df.dtypes

Rows, Cols: (99441, 8)


order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [2]:
df.head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00


In [3]:
missing = df.isnull().sum()
missing_pct = df.isnull().mean() * 100
pd.DataFrame({'Count': missing, 'Pct': missing_pct}).sort_values(by='Count', ascending=False)

,Count,Pct
order_delivered_customer_date,2965,2.981668
order_delivered_carrier_date,1783,1.793023
order_approved_at,160,0.160899
order_id,0,0.000000
order_purchase_timestamp,0,0.000000
order_status,0,0.000000
customer_id,0,0.000000
order_estimated_delivery_date,0,0.000000


In [4]:
print('Full row duplicates:', df.duplicated().sum())

Full row duplicates: 0


In [5]:
print('order_id duplicates:', df['order_id'].duplicated().sum())
print('Row count == distinct order_id:', len(df) == df['order_id'].nunique())

order_id duplicates: 0
Row count == distinct order_id: True


In [6]:
df['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [7]:
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

# Cross-tab null pattern against order_status
df.groupby('order_status')[date_cols].apply(lambda x: x.isnull().sum())

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,
approved,0,0,2,2,0
canceled,0,141,550,619,0
created,0,5,5,5,0
delivered,0,14,2,8,0
invoiced,0,0,314,314,0
processing,0,0,301,301,0
shipped,0,0,0,1107,0
unavailable,0,0,609,609,0


In [8]:
df['time_to_approval'] = (df['order_approved_at'] - df['order_purchase_timestamp']).dt.total_seconds() / 3600
df['time_to_carrier'] = (df['order_delivered_carrier_date'] - df['order_approved_at']).dt.total_seconds() / 3600
df['time_to_delivery'] = (df['order_delivered_customer_date'] - df['order_delivered_carrier_date']).dt.total_seconds() / 3600
df['delivery_delay'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.total_seconds() / (3600*24)

df[['time_to_approval', 'time_to_carrier', 'time_to_delivery', 'delivery_delay']].describe()

,time_to_approval,time_to_carrier,time_to_delivery,delivery_delay
count,99281.000000,97644.000000,96475.000000,96476.000000
mean,10.419094,67.320916,223.933126,-11.179120
std,26.038004,85.186249,210.242923,10.186113
min,0.000000,-4109.256111,-386.308056,-146.016123
25%,0.215000,21.012222,98.398750,-16.244384
50%,0.343333,43.641528,170.394444,-11.948941
75%,14.580833,85.931250,288.698750,-6.390000
max,4509.180556,3018.301667,4924.583333,188.975081


### Takeaways
- Orders table has 99,441 rows. Most orders (96,478) are successfully 'delivered'.
- Dates are correctly missing primarily for canceled or unavailable orders, reflecting their real-world status.
- The primary key `order_id` is unique and has no duplicates. Some orders arrive earlier than estimated (negative delivery delay).